In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import numpy as np
import json

# Load the s1K dataset
ds = load_dataset("simplescaling/s1K", split="train")

# Extract questions 
questions = [sample.get("question", "") for sample in ds]
print(f"Found {len(questions)} questions.")

# Compute embeddings using SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(
    questions,
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True,  
)

# === Perform KMeans clustering ===
n_clusters = 20 
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(embeddings)

# === Save results as JSONL ===
output_path = "train.jsonl"
print(f"Saving clustered dataset to {output_path}...")
with open(output_path, "w", encoding="utf-8") as f:
    for i, (sample, cluster_id) in enumerate(zip(ds, cluster_ids)):
        item = {
            "idx": i,
            "sentence": sample.get("question", ""),
            "solution": sample.get("solution", ""),
            "cluster_id": str(cluster_id),
        }
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")

print(f"File saved: {output_path}")

In [82]:
!sbatch /leonardo/home/userexternal/miacobel/project_new/ACM/acm.sh

Submitted batch job 24795621


In [6]:
!sbatch /leonardo/home/userexternal/miacobel/project_new/Qwen2.5-Math/evaluation/sh/qwen_eval.sh

Submitted batch job 25023841


In [ ]:
!python /leonardo/home/userexternal/miacobel/project_new/Qwen2.5-Math/evaluation/extract_result.py \
  --root_folder /leonardo/home/userexternal/miacobel/project_new/Qwen2.5-Math/evaluation/outputs/ACM-TA-1.5B \
  --model_name_or_path deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B